# STAT3612 数据管线（合并版）

本 Notebook 将以下内容合并为一条可交互执行链路：

- `scripts/data_utils.py`：读 JSON / CSV、按 `case_id` 合并、`image_path` 修正
- `scripts/build_table.py`：统一大表 + 数据审计打印
- `scripts/baseline.py`：结构化 + radiomics 的 Logistic Regression baseline + 提交文件
- `docs/data_quality_report.md`：数据质量报告提纲（见下文 Markdown 区，运行后把打印结果填回 `docs/` 亦可）

**运行目录**：从仓库根目录启动 Jupyter，或从 `notebooks/` 启动均可；下方首个代码格会自动 `chdir` 到含 `kaggle-dataset` 的那一层。

**依赖**：`pip install pandas numpy scikit-learn jupyter`

---

## 文档：Data Quality Report 提纲（对应 `docs/data_quality_report.md`）

### 1. 基本信息

- **train 样本数**：（运行「数据审计」后填写）
- **val 样本数**：
- **test 样本数**：
- **主键列**：`case_id`

### 2. 各 split 统一表信息

将下方「数据审计」单元格的 **stdout** 复制到团队 `docs/data_quality_report.md` 对应小节。

### 3. 关键字段质量摘要

- **Age / age**：缺失率、填补策略
- **Sex / sex**：类别、unknown/None 处理
- **Tumor Location**：Top-N、拼写统一
- **Radiomics**：异常值、常数列、是否标准化

### 4. ID 对齐

- JSON vs `clinical_information`
- JSON vs `original_raw_report`

### 5. 结论与风险

- 可用字段、泄漏风险、类别不平衡等

## 0. 环境与仓库根目录

将 `scripts` 加入 `sys.path`，并把 `data_utils.DATA_DIR` 指到 `.../kaggle-dataset`。

In [1]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
ROOT = _here if (_here / "kaggle-dataset").is_dir() else _here.parent
if not (ROOT / "kaggle-dataset").is_dir():
    raise FileNotFoundError(f"未找到 kaggle-dataset，当前 ROOT={ROOT}")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))

import data_utils as du

du.DATA_DIR = str(ROOT / "kaggle-dataset")
print("ROOT =", ROOT)
print("DATA_DIR =", du.DATA_DIR)

ROOT = /Users/sunnie/Desktop/STAT3612-Amateur-Neuro-Team
DATA_DIR = /Users/sunnie/Desktop/STAT3612-Amateur-Neuro-Team/kaggle-dataset


## 1. 数据审计（`build_table` 逻辑）

对 `train` / `val` / `test` 调用 `merge_all_sources`，打印 shape、缺失率 Top20、`Overall_class` 分布（若有）。

In [2]:
from typing import List

import pandas as pd


def audit_split(split: str, top_missing: int = 20) -> pd.DataFrame:
    df = du.merge_all_sources(split)

    print(f"\n[{split}] 统一表信息")
    print("-" * 40)
    print("shape:", df.shape)
    print("case_id 行数:", len(df), "唯一个数:", df["case_id"].nunique())

    miss = df.isna().mean().sort_values(ascending=False).head(top_missing)
    print("\n缺失率最高的前若干列：")
    print(miss.to_string())

    if "Overall_class" in df.columns:
        print("\n标签分布 Overall_class：")
        print(df["Overall_class"].value_counts(dropna=False))

    return df


dfs = {}
for s in ["train", "val", "test"]:
    dfs[s] = audit_split(s)


[train] 统一表信息
----------------------------------------
shape: (1983, 50)
case_id 行数: 1983 唯一个数: 1983

缺失率最高的前若干列：
flag                                   0.986889
location                               0.852748
ax_t1c__age                            0.697932
ax_t1__sex                             0.697932
ax_t1c__sex                            0.697932
ax_t2__sex                             0.697932
ax_t2__age                             0.697932
ax_t1__age                             0.697932
ax_t2f__age                            0.697932
Age_y                                  0.697932
ax_t2f__sex                            0.697932
Sex_x                                  0.695915
Age_x                                  0.695915
ax_t2f__rad_glcm_JointEntropy          0.361069
ax_t2f__rad_firstorder_Mean            0.361069
ax_t2f__rad_firstorder_Entropy         0.361069
ax_t2f__rad_firstorder_90Percentile    0.361069
ax_t2f__rad_glcm_Contrast              0.361069
ax_t1c__rad_firstorde

## 2.（可选）抽查一条 `image_path` 与 `.npy` 形状

确认路径映射 `image_features/` → `image_features/image_features/` 在本机可用。

In [3]:
import numpy as np

sample = dfs["train"].iloc[0]
cid = sample["case_id"]
paths = sample.get("image_path", [])
if isinstance(paths, list) and len(paths) > 0:
    p0 = paths[0]
    arr = du.load_image_feature_vector(p0)
    print("case_id:", cid)
    print("image_path[0]:", p0)
    print("npy shape:", arr.shape, "dtype:", arr.dtype)
else:
    print("该样本无 image_path 列表，跳过")

case_id: 7
image_path[0]: image_features/7/ax_t1/image.npy
npy shape: (2048,) dtype: float32


## 3. Baseline 训练、验证与提交（`baseline.py` 逻辑）

使用结构化临床字段 + 全部 radiomics 列；验证集指标为 Accuracy 与 F1-macro；预测写入项目根下 `outputs/sample_submission_v1.csv`（相对于 `ROOT`）。

In [4]:
import baseline as bl

model = bl.train_and_eval()
out = ROOT / "outputs" / "sample_submission_v1.csv"
out.parent.mkdir(parents=True, exist_ok=True)
bl.predict_test_and_save(model, output_path=str(out))

[train] 使用特征列数: 25
[val] 使用特征列数: 25
数值特征列数: 20 类别特征列数: 5
开始训练 baseline 模型...
在 val 集上评估...
Val Accuracy = 0.6820
Val F1-macro = 0.5638
[test] 使用特征列数: 25
在 test 集上生成预测...
提交文件已保存到: /Users/sunnie/Desktop/STAT3612-Amateur-Neuro-Team/outputs/sample_submission_v1.csv


## 4. 说明

- 命令行仍可使用：`python scripts/build_table.py`、`python scripts/baseline.py`，与 Notebook 共用同一套 `scripts/`。
- 若希望 **完全脱离** `scripts/`（单文件复制粘贴给朋友），可把 `data_utils.py` / `baseline.py` 全文贴进新代码格并去掉 `import baseline` — 当前设计优先避免三处重复维护。